# 02 — Crop Recommendation: Model Training & Comparison
Trains RandomForest, LightGBM, and XGBoost on the real Crop_recommendation.csv dataset, compares them with cross-validation, and saves the best model.

This notebook mirrors `utils/train_models.py::train_crop_model` — run that script to regenerate the production `.pkl` files used by the Streamlit app.

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42

In [ ]:
df = pd.read_csv('../dataset/Crop_recommendation.csv')
feature_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
X = df[feature_cols]
le = LabelEncoder()
y = le.fit_transform(df['label'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
X_train.shape, X_test.shape

## Train candidate models

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)

lgbm = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                       random_state=RANDOM_STATE, verbose=-1)
lgbm.fit(X_train, y_train)

xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                     random_state=RANDOM_STATE, eval_metric='mlogloss')
xgb.fit(X_train, y_train)

models = {'RandomForest': rf, 'LightGBM': lgbm, 'XGBoost': xgb}

## Cross-validation & test-set comparison

In [ ]:
results = []
for name, model in models.items():
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    preds = model.predict(X_test)
    results.append({
        'model': name,
        'cv_accuracy_mean': cv.mean(),
        'test_accuracy': accuracy_score(y_test, preds),
        'test_f1_macro': f1_score(y_test, preds, average='macro'),
    })
results_df = pd.DataFrame(results)
results_df

## Confusion matrix — best model

In [ ]:
best_row = results_df.sort_values('test_accuracy', ascending=False).iloc[0]
best_model = models[best_row['model']]
print('Best model:', best_row['model'])
print(classification_report(y_test, best_model.predict(X_test), target_names=le.classes_))

## Feature importance

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importance = pd.Series(best_model.feature_importances_, index=feature_cols).sort_values()
    importance.plot(kind='barh', figsize=(6, 4), color='#2E7D32')

## Save artifacts
The production copies are written by `utils/train_models.py`; this cell shows the equivalent save step for reference.

In [ ]:
# joblib.dump(best_model, '../models/crop_model.pkl')
# joblib.dump(le, '../models/crop_label_encoder.pkl')
print('See utils/train_models.py for the production save step.')